# PharmaPulse Exploratory Data Analysis

## 1. Notebook Objective

This notebook explores the cleaned PharmaPulse datasets to understand revenue performance, HCP engagement, medical rep productivity, campaign response, and product margin contribution. The analysis uses cleaned CSV files so it can be run locally without requiring a database connection.

## 2. Load Cleaned Datasets

This section loads the eight cleaned datasets from `data/cleaned` and prepares a few date columns for analysis.

In [ ]:
import os
import tempfile
from pathlib import Path

current_path = Path.cwd()
if (current_path / "data" / "cleaned").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "sample_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MATPLOTLIB_CACHE_DIR = Path(tempfile.gettempdir()) / "pharmapulse_matplotlib_cache"
MATPLOTLIB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MATPLOTLIB_CACHE_DIR)

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.float_format", "{:,.2f}".format)

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

territories = pd.read_csv(CLEANED_DIR / "territories.csv")
hcps = pd.read_csv(CLEANED_DIR / "hcps.csv")
medical_reps = pd.read_csv(CLEANED_DIR / "medical_reps.csv", parse_dates=["hire_date"])
products = pd.read_csv(CLEANED_DIR / "products.csv", parse_dates=["launch_date"])
hcp_calls = pd.read_csv(
    CLEANED_DIR / "hcp_calls.csv",
    parse_dates=["call_date", "next_call_planned_date"],
)
sales = pd.read_csv(CLEANED_DIR / "sales.csv", parse_dates=["sale_date"])
campaigns = pd.read_csv(CLEANED_DIR / "campaigns.csv", parse_dates=["start_date", "end_date"])
campaign_engagement = pd.read_csv(
    CLEANED_DIR / "campaign_engagement.csv",
    parse_dates=["engagement_date"],
)

datasets = {
    "territories": territories,
    "hcps": hcps,
    "medical_reps": medical_reps,
    "products": products,
    "hcp_calls": hcp_calls,
    "sales": sales,
    "campaigns": campaigns,
    "campaign_engagement": campaign_engagement,
}

print("Cleaned datasets loaded successfully.")

## Data Quality Reminder

The cleaned datasets used in this notebook were already validated before this analysis. The validation process checked required files, required columns, primary keys, foreign keys, numeric ranges, HCP tier values, date ranges, and territory consistency.

## 3. Basic Dataset Overview and Row Counts

A row-count overview confirms the expected analytical coverage across dimensions and activity tables.

In [ ]:
row_counts = pd.DataFrame(
    {
        "table_name": list(datasets.keys()),
        "row_count": [len(dataframe) for dataframe in datasets.values()],
        "column_count": [dataframe.shape[1] for dataframe in datasets.values()],
    }
)

row_counts

## 4. Revenue Distribution

This section reviews the distribution of `net_sales`, which is the primary revenue metric for the project.

In [ ]:
revenue_summary = sales["net_sales"].describe().to_frame(name="net_sales")
display(revenue_summary)

plt.figure(figsize=(9, 5))
plt.hist(sales["net_sales"], bins=30)
plt.title("Revenue Distribution by Sales Transaction")
plt.xlabel("Net Sales")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

## 5. Revenue by Territory

Territory revenue highlights where commercial performance is strongest and where performance may require review.

In [ ]:
revenue_by_territory = (
    sales.merge(territories, on="territory_id")
    .groupby(["territory_id", "territory_name", "region", "city"], as_index=False)
    .agg(
        total_net_sales=("net_sales", "sum"),
        unique_hcps=("hcp_id", "nunique"),
        sales_transactions=("sales_id", "count"),
    )
    .sort_values("total_net_sales", ascending=False)
)

revenue_by_territory.to_csv(OUTPUT_DIR / "eda_revenue_by_territory.csv", index=False)
display(revenue_by_territory)

plt.figure(figsize=(10, 5))
plt.bar(revenue_by_territory["territory_name"], revenue_by_territory["total_net_sales"])
plt.title("Net Sales by Territory")
plt.xlabel("Territory")
plt.ylabel("Total Net Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Revenue by Therapy Area

Therapy-area revenue shows which parts of the product portfolio contribute the most commercial value.

In [ ]:
revenue_by_therapy_area = (
    sales.merge(products, on="product_id")
    .groupby("therapy_area", as_index=False)
    .agg(
        total_net_sales=("net_sales", "sum"),
        total_units_sold=("units_sold", "sum"),
        unique_hcps=("hcp_id", "nunique"),
        products_sold=("product_id", "nunique"),
    )
    .sort_values("total_net_sales", ascending=False)
)

display(revenue_by_therapy_area)

plt.figure(figsize=(9, 5))
plt.bar(revenue_by_therapy_area["therapy_area"], revenue_by_therapy_area["total_net_sales"])
plt.title("Net Sales by Therapy Area")
plt.xlabel("Therapy Area")
plt.ylabel("Total Net Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. HCP Tier Performance

HCP tier analysis compares revenue, call activity, and engagement across commercial priority segments.

In [ ]:
sales_by_hcp = (
    sales.groupby("hcp_id", as_index=False)
    .agg(total_net_sales=("net_sales", "sum"), sales_transactions=("sales_id", "count"))
)

calls_by_hcp = (
    hcp_calls.groupby("hcp_id", as_index=False)
    .agg(
        total_calls=("call_id", "count"),
        avg_engagement_score=("engagement_score", "mean"),
    )
)

hcp_performance = (
    hcps.merge(sales_by_hcp, on="hcp_id", how="left")
    .merge(calls_by_hcp, on="hcp_id", how="left")
)
hcp_performance["total_net_sales"] = hcp_performance["total_net_sales"].fillna(0)
hcp_performance["sales_transactions"] = hcp_performance["sales_transactions"].fillna(0)
hcp_performance["total_calls"] = hcp_performance["total_calls"].fillna(0)

hcp_tier_performance = (
    hcp_performance.groupby("hcp_tier", as_index=False)
    .agg(
        hcp_count=("hcp_id", "nunique"),
        total_net_sales=("total_net_sales", "sum"),
        avg_revenue_per_hcp=("total_net_sales", "mean"),
        avg_calls_per_hcp=("total_calls", "mean"),
        avg_engagement_score=("avg_engagement_score", "mean"),
    )
    .sort_values("total_net_sales", ascending=False)
)

hcp_tier_performance.to_csv(OUTPUT_DIR / "eda_hcp_tier_performance.csv", index=False)
display(hcp_tier_performance)

plt.figure(figsize=(8, 5))
plt.bar(hcp_tier_performance["hcp_tier"], hcp_tier_performance["avg_revenue_per_hcp"])
plt.title("Average Revenue per HCP by Tier")
plt.xlabel("HCP Tier")
plt.ylabel("Average Net Sales per HCP")
plt.tight_layout()
plt.show()

## 8. Rep Target Achievement

Rep target achievement compares actual net sales against assigned sales targets.

In [ ]:
sales_by_rep = (
    sales.groupby("rep_id", as_index=False)
    .agg(total_net_sales=("net_sales", "sum"), active_hcps=("hcp_id", "nunique"))
)

rep_target_achievement = (
    medical_reps.merge(sales_by_rep, on="rep_id", how="left")
    .merge(territories[["territory_id", "territory_name"]], on="territory_id", how="left")
)
rep_target_achievement["total_net_sales"] = rep_target_achievement["total_net_sales"].fillna(0)
rep_target_achievement["active_hcps"] = rep_target_achievement["active_hcps"].fillna(0)
rep_target_achievement["target_achievement_percent"] = (
    rep_target_achievement["total_net_sales"] / rep_target_achievement["sales_target"] * 100
)
rep_target_achievement["target_status"] = rep_target_achievement["target_achievement_percent"].apply(
    lambda value: "At or above target" if value >= 100 else "Below target"
)

rep_target_achievement = rep_target_achievement.sort_values(
    "target_achievement_percent", ascending=False
)
rep_target_achievement.to_csv(OUTPUT_DIR / "eda_rep_target_achievement.csv", index=False)
display(rep_target_achievement.head(10))

plt.figure(figsize=(10, 5))
plt.bar(rep_target_achievement["rep_name"].head(10), rep_target_achievement["target_achievement_percent"].head(10))
plt.title("Top 10 Reps by Target Achievement")
plt.xlabel("Medical Rep")
plt.ylabel("Target Achievement Percent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 9. Call Volume vs Revenue

This section compares completed HCP call volume with rep-level revenue. The relationship is interpreted as an association, not as a causal result.

In [ ]:
completed_calls_by_rep = (
    hcp_calls[hcp_calls["call_status"] == "Completed"]
    .groupby("rep_id", as_index=False)
    .agg(completed_calls=("call_id", "count"), avg_engagement_score=("engagement_score", "mean"))
)

call_volume_vs_revenue = medical_reps[["rep_id", "rep_name"]].merge(
    completed_calls_by_rep, on="rep_id", how="left"
).merge(sales_by_rep, on="rep_id", how="left")

call_volume_vs_revenue[["completed_calls", "total_net_sales", "active_hcps"]] = call_volume_vs_revenue[
    ["completed_calls", "total_net_sales", "active_hcps"]
].fillna(0)

display(call_volume_vs_revenue.sort_values("total_net_sales", ascending=False).head(10))

plt.figure(figsize=(8, 5))
plt.scatter(call_volume_vs_revenue["completed_calls"], call_volume_vs_revenue["total_net_sales"])
plt.title("Completed Calls vs Net Sales by Rep")
plt.xlabel("Completed HCP Calls")
plt.ylabel("Total Net Sales")
plt.tight_layout()
plt.show()

## 10. Campaign Engagement Overview

Campaign engagement metrics summarize HCP response patterns by territory.

In [ ]:
campaign_engagement_overview = (
    campaign_engagement.groupby("territory_id", as_index=False)
    .agg(
        total_engagements=("engagement_id", "count"),
        opens=("opened", "sum"),
        clicks=("clicked", "sum"),
        attendances=("attended", "sum"),
        avg_response_score=("response_score", "mean"),
    )
    .merge(territories[["territory_id", "territory_name", "region"]], on="territory_id")
)
campaign_engagement_overview["open_rate_percent"] = (
    campaign_engagement_overview["opens"] / campaign_engagement_overview["total_engagements"] * 100
)
campaign_engagement_overview["click_rate_percent"] = (
    campaign_engagement_overview["clicks"] / campaign_engagement_overview["total_engagements"] * 100
)
campaign_engagement_overview["attendance_rate_percent"] = (
    campaign_engagement_overview["attendances"] / campaign_engagement_overview["total_engagements"] * 100
)

campaign_engagement_overview = campaign_engagement_overview.sort_values(
    "avg_response_score", ascending=False
)
display(campaign_engagement_overview)

plt.figure(figsize=(10, 5))
plt.bar(campaign_engagement_overview["territory_name"], campaign_engagement_overview["avg_response_score"])
plt.title("Average Campaign Response Score by Territory")
plt.xlabel("Territory")
plt.ylabel("Average Response Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Product Margin Contribution

Estimated margin contribution combines net sales with product margin percentage to identify products that may contribute the most commercial value.

In [ ]:
sales_with_products = sales.merge(products, on="product_id")
sales_with_products["estimated_margin_contribution"] = (
    sales_with_products["net_sales"] * sales_with_products["margin_percent"] / 100
)

product_margin_contribution = (
    sales_with_products.groupby(["product_id", "product_name", "therapy_area", "margin_percent"], as_index=False)
    .agg(
        total_net_sales=("net_sales", "sum"),
        estimated_margin_contribution=("estimated_margin_contribution", "sum"),
        units_sold=("units_sold", "sum"),
    )
    .sort_values("estimated_margin_contribution", ascending=False)
)

product_margin_contribution.to_csv(OUTPUT_DIR / "eda_product_margin_contribution.csv", index=False)
display(product_margin_contribution.head(10))

plt.figure(figsize=(10, 5))
plt.bar(product_margin_contribution["product_name"].head(10), product_margin_contribution["estimated_margin_contribution"].head(10))
plt.title("Top 10 Products by Estimated Margin Contribution")
plt.xlabel("Product")
plt.ylabel("Estimated Margin Contribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 12. Key Observations and Next Steps

Based on the notebook outputs:

- Total net sales are approximately 1,737.6M across 17,967 sales transactions.
- Pune Central is the highest-revenue territory at approximately 204.5M, while Ahmedabad Gujarat is lowest at approximately 119.6M and may require review.
- Respiratory is the largest therapy area by revenue at approximately 322.3M, indicating a major portfolio contribution.
- Tier 1 HCPs generate the highest total revenue among tiers, with 111 HCPs and average engagement near 50.6.
- PainAway has the highest estimated margin contribution at approximately 146.2M, suggesting it is a priority product for margin-focused reporting.

Suggested next steps:

- Review territory-level differences in revenue and HCP coverage.
- Compare rep target achievement with call volume and active HCP coverage.
- Use HCP tier, engagement, and revenue signals to support call planning.
- Evaluate campaign response metrics by territory before scaling similar campaigns.